# J.A.R.V.I.S. — QLoRA Fine-Tune (Offloaded Model Evolution)

Fine-tuning **NEVER happens on the Helio G85 phone**. It happens here, on a
cloud **T4 GPU** (free-tier Google Colab or Kaggle). This notebook:

1. Loads an **encrypted dataset** (uploaded from the phone via Tailscale/SCP, or pulled from a private source).
2. Fine-tunes **Qwen2.5-7B (Oracle edge)** or **Qwen2.5-1.5B (phone)** with **Unsloth + QLoRA** (memory-efficient).
3. Validates against a **holdout set** before allowing deployment.
4. Exports the adapter **`.safetensors`** and prints an SHA-256 checksum.
5. Emits the `sync_adapter_to_edge.sh` command for the operator to run.

> ⚙️ Runtime: **GPU T4 / T4 ×2 / A100**. Install deps below, then run top-to-bottom.

In [ ]:
# 1. Install Unsloth (fastest QLoRA on consumer/Colab GPUs)
!pip install -q unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git
!pip install -q --no-warn-script-location torch xformers trl peft accelerate bitsandbytes

In [ ]:
import os, json, hashlib, base64, subprocess

# ---- Configuration (EDIT ME) ----
BASE_MODEL = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
ADAPTER_NAME = os.environ.get("ADAPTER_NAME", "jarvis-qwen-1.5b-v1")
DATASET_ENCRYPTED = os.environ.get("DATASET_ENCRYPTED", "data/dataset.json.enc")
DATASET_KEY = os.environ.get("DATASET_KEY", "")   # AES passphrase put here or in secrets
OUT_DIR = f"./adapter-{ADAPTER_NAME}"
MAX_SEQ = int(os.environ.get("MAX_SEQ", "1024"))
EPOCHS = int(os.environ.get("EPOCHS", "2"))
LR = float(os.environ.get("LR", "2e-4"))

os.makedirs("data", exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Config: {BASE_MODEL} / {ADAPTER_NAME} / max_seq={MAX_SEQ}")

## Decrypt the training dataset

The training data is encrypted **on the phone** before it leaves the device
(AES-256-GCM via `utils/sovereign_terminal.encrypt_outgoing`). Here we decrypt
locally in Colab memory and never re-upload plaintext back to the mesh.

Expected format on disk: `{"ciphertext": "<b64>", "iv": "<b64>", "mac": "<hex>", "ts": 1234}`,
as produced by `utils/device_comm.encrypt_payload`.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
import base64 as _b64

def _derive_key(passphrase: str, salt: bytes = b"jarvis-l7") -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=120_000)
    return kdf.derive(passphrase.encode())

def decrypt_dataset(path: str, passphrase: str) -> list:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Encrypted dataset not found: {path}")
    with open(path) as fh:
        env = json.load(fh)
    key = _derive_key(passphrase or os.environ.get("DATASET_KEY", ""))
    ct = _b64.urlsafe_b64decode(env["ct"])
    iv = _b64.urlsafe_b64decode(env["iv"])
    raw = AESGCM(key).decrypt(iv, ct, None)
    if env.get("gzip"):
        import zlib; raw = zlib.decompress(raw)
    return json.loads(raw.decode("utf-8"))

dataset = decrypt_dataset(DATASET_ENCRYPTED, DATASET_KEY)
print(f"Decrypted {len(dataset)} training examples")
print(json.dumps(dataset[0] if dataset else {}, ensure_ascii=False)[:400])

In [ ]:
# ---- Split holdout (validation) set, alignment-format into Unsloth ----
from sklearn.model_selection import train_test_split
import random

train, valid = train_test_split(dataset, test_size=0.1, random_state=42)

def to_chat(item):
    return [
        {"role": "system", "content": "Anda adalah J.A.R.V.I.S., asisten AI personal."},
        {"role": "user", "content": item.get("instruction", "") or item.get("input", "")},
        {"role": "assistant", "content": item.get("output", "")},
    ]

# Save validation JSONL for post-training evaluation
with open("data/validation.jsonl", "w") as fh:
    for item in valid:
        fh.write(json.dumps(to_chat(item), ensure_ascii=False) + "\n")

train_messages = [to_chat(i) for i in train]
print(f"train={len(train_messages)} valid={len(valid)} samples ready")

## Load base model with Unsloth + QLoRA (4-bit)

Unsloth loads QLoRA in 4-bit so we fit 7B on a single T4 (16GB). The phone
target (1.5B) trains in seconds and the adapter stays small enough to ship.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("QLoRA adapter attached.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

train_ds = Dataset.from_list([{"messages": m} for m in train_messages])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="messages",
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUT_DIR,
        report_to="none",
    ),
)
trainer.train()
print("Training complete.")

## Validate on the holdout set

Before we allow deployment, score the adapter against `data/validation.jsonl`.
If average loss exceeds a threshold the cell reports FAIL and stops the export.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

losses = []
model.eval()
with torch.no_grad():
    with open("data/validation.jsonl") as fh:
        for line in fh:
            msgs = json.loads(line)
            target = msgs[-1]["content"]
            prompt_txt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt_txt + " " + target, return_tensors="pt").to("cuda")
            labels = inputs["input_ids"].clone()
            prompt_len = len(tokenizer(prompt_txt, return_tensors="pt")["input_ids"][0])
            labels[0, :prompt_len] = -100
            out = model(input_ids=inputs["input_ids"], labels=labels)
            losses.append(out.loss.item())

mean_loss = float(np.mean(losses))
VALIDATION_THRESHOLD = float(os.environ.get("VALIDATION_THRESHOLD", "1.5"))
print(f"Validation loss: {mean_loss:.3f} (threshold {VALIDATION_THRESHOLD})")
if mean_loss > VALIDATION_THRESHOLD:
    raise SystemExit("HOLD-OUT VALIDATION FAILED — do not deploy.")
print(f"Validation passed. loss_valid={mean_loss:.3f}")

## Export the adapter (`.safetensors`) + SHA-256

The exported PEFT adapter is small (a few MB for 1.5B). We print its SHA-256
so the `sync_adapter_to_edge.sh` script can verify integrity before hot-swap.

In [ ]:
model.save_pretrained(f"{OUT_DIR}/safetensors")
tokenizer.save_pretrained(f"{OUT_DIR}/safetensors")
print("Saved adapter:", OUT_DIR)

# Compute SHA-256 of the adapter payload (assuming single safetensors for 1.5B)
import glob
adapter_files = glob.glob(f"{OUT_DIR}/**/*.safetensors", recursive=True)
for f in adapter_files:
    h = hashlib.sha256()
    with open(f, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    print(f"SHA-256 {h.hexdigest()}  {f}")

# Save metadata consumed by sync script
meta = {"name": ADAPTER_NAME, "base": BASE_MODEL, "loss_valid": round(mean_loss, 4),
        "sha256": (h.hexdigest() if adapter_files else ""),
        "adapter_dir": OUT_DIR}
with open(f"{OUT_DIR}/adapter_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print(json.dumps(meta, indent=2))

## Sync command (run on your laptop / Oracle edge)

Copy the adapter off Colab to the Oracle edge / phone. Use HTTPS / SCP over
Tailscale and verify the SHA-256. FYI the hot-swap + rollback is handled by
`scripts/sync_adapter_to_edge.sh` — run it from this machine's repo checkout:

In [ ]:
# If you downloaded this notebook's output to a local repo checkout:
#   cd /path/to/jarvis
#   scripts/sync_adapter_to_edge.sh \
#       --adapter "<local adapter dir>" \
#       --target oracle \
#       --host <oracle-tailscale-ip> --user ubuntu
#
print("Done. Run scripts/sync_adapter_to_edge.sh to deploy this adapter.")